In [2]:
import duckdb

In [2]:
with duckdb.connect('files.duckdb') as con:
    # Set configuration for better performance

    # More efficient preview
    con.table("stg_transponderOwnership").show()

┌─────────────┬────────┐
│ transponder │ share  │
│   boolean   │ double │
├─────────────┼────────┤
│ true        │  0.184 │
│ false       │  0.816 │
└─────────────┴────────┘



In [ ]:
with duckdb.connect('/files.duckdb') as con:
    con.sql("""
            
with abm3_source as (
    select
        case when transponder_ownership then 'Yes' else 'No' end as transponder_ownership,
        unittype,
        case 
            when unittype = 1 then 
                sum(count(*)) over (partition by transponder_ownership) * 1.0 / 
                sum(count(*)) over ()
            else 
                count(*) * 1.0 / 
                sum(count(*)) over (partition by unittype)
        end as proportion
    from files.stg_households
    group by transponder_ownership, unittype
),

hts_source as (
    select 
        case when transponder then 'Yes' else 'No' end as transponder_ownership,  
        share as proportion 
    FROM files.stg_transponderOwnership
),

hts_abm3_joined as (
    select 
        a.transponder_ownership, 
        a.unittype,
        a.proportion as abm_proportion, 
        h.proportion as hts_proportion
    from abm3_source a
    join hts_source h on a.transponder_ownership = h.transponder_ownership
)

select * from hts_abm3_joined where transponder_ownership = 'Yes'

""").show()

┌───────────────────────┬──────────┬─────────────────────┬────────────────┐
│ transponder_ownership │ unittype │   abm_proportion    │ hts_proportion │
│        varchar        │  int64   │       double        │     double     │
├───────────────────────┼──────────┼─────────────────────┼────────────────┤
│ Yes                   │        0 │ 0.23212710000758313 │          0.184 │
│ Yes                   │        1 │ 0.21324584946310665 │          0.184 │
└───────────────────────┴──────────┴─────────────────────┴────────────────┘



In [5]:
with duckdb.connect('../files.duckdb') as con:

    con.sql("SELECT unittype, count(*) FROM stg_households GROUP BY unittype").show()

┌──────────┬──────────────┐
│ unittype │ count_star() │
│  int64   │    int64     │
├──────────┼──────────────┤
│        0 │      1160472 │
│        1 │       116411 │
└──────────┴──────────────┘



In [9]:
with duckdb.connect('../files.duckdb') as con:

    con.sql(""" with abm3_source as (
    select 
        case when transponder_ownership then 'Yes' else 'No' end as transponder_ownership,
        'non-GQ' as household_type,
        count(*) * 1.0 / sum(count(*)) over () as proportion
    from stg_households
    where unittype = 0 --non-GQ only
    group by transponder_ownership, household_type
    UNION ALL
    select 
        case when transponder_ownership then 'Yes' else 'No' end as transponder_ownership,
        'all households (includes GQ)' as household_type,
        count(*) * 1.0 / sum(count(*)) over () as proportion
    from stg_households
    --where unittype = 0 --all households including GQ
    group by transponder_ownership,household_type
),

hts_source as (
    select 
        case when transponder then 'Yes' else 'No' end as transponder_ownership,  
        share as proportion 
    FROM stg_transponderOwnership
),

hts_abm3_joined as (
    select 
        a.transponder_ownership, 
        a.household_type,
        a.proportion as abm_proportion, 
        h.proportion as hts_proportion
    from abm3_source a
    join hts_source h on a.transponder_ownership = h.transponder_ownership
)

select * from hts_abm3_joined where transponder_ownership = 'Yes'
            """).show()

┌───────────────────────┬──────────────────────────────┬─────────────────────┬────────────────┐
│ transponder_ownership │        household_type        │   abm_proportion    │ hts_proportion │
│        varchar        │           varchar            │       double        │     double     │
├───────────────────────┼──────────────────────────────┼─────────────────────┼────────────────┤
│ Yes                   │ non-GQ                       │ 0.23212710000758313 │          0.184 │
│ Yes                   │ all households (includes GQ) │ 0.23212710000758313 │          0.184 │
└───────────────────────┴──────────────────────────────┴─────────────────────┴────────────────┘



In [ ]:
select * from hts_abm3_joined where transponder_ownership = 'Yes'

In [ ]:
with duckdb.connect('../files.duckdb') as con:
    # Set configuration for better performance

    # More efficient preview
    con.sql("SELECT filename FROM stg_trips").show()

BinderException: Binder Error: Referenced column "filename" not found in FROM clause!
Candidate bindings: "distance_mm", "value_of_time_category_id", "time_transit_in_vehicle", "trip_veh_age", "distance_bike"

In [8]:
with duckdb.connect('files.duckdb') as con:
    con.sql("SELECT * FROM stg_trips ORDER BY trip_id ASC").show()


┌─────────┬───────────┬──────────────┬─────────────────┬──────────┬──────────┬────────────┬─────────────┬────────┬─────────┬─────────────────────┬─────────────────────────┬──────────┬────────────────────┬────────┬────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬──────────────────┬─────────────┬────────────────────┬───────────────┬───────────────────────┬────────────────────┬──────────────┬──────────────┬────────────────────────────┬────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬────────────┬───────────────┬───────────────────┬───────────────────┬────────────────────────┬────────────────────────┬───────────────────┬──────────┬──────────┬───────────────┬───────────────┬───────────┬─────────────────────────────────────────┬───────────────────────────────────────┬───────────────────────────────────┬────────────────────────────────┬──────────────────────────────┬──────────────────────────┬────────

In [16]:
# Connect to DuckDB
con = duckdb.connect('files.duckdb')

# Install extensions
con.execute("INSTALL azure;")
con.execute("LOAD azure;")
con.execute("INSTALL delta;")
con.execute("LOAD delta;")

# Set up Azure authentication
con.execute("""
    CREATE OR REPLACE SECRET azure_secret (
        TYPE AZURE,
        CONNECTION_STRING 'AccountName=adlsdasadsdevwest;SharedAccessSignature=***REDACTED***'
    );
""")

# Read from Delta table
delta_table_path = "***REDACTED***"

# Query the Delta table
result = con.sql(f"""
    SELECT * 
    FROM delta_scan('{delta_table_path}')
""").show()

con.close()

#print(result)

# # Or load into a DuckDB table
# con.execute(f"""
#     CREATE TABLE local_table AS 
#     SELECT * FROM delta_scan('{delta_table_path}')
# """)

┌─────────────┬─────────────┬────────────┬────────┬────────────────────────┬───────────────────────────────┐
│ scenario_id │    model    │ trip_mode  │ count  │ loader__delta_hash_key │     loader__updated_date      │
│    int32    │   varchar   │  varchar   │ int64  │         int32          │   timestamp with time zone    │
├─────────────┼─────────────┼────────────┼────────┼────────────────────────┼───────────────────────────────┤
│        1150 │ visitor     │ WALK       │  62558 │            -1813879661 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ visitor     │ TNC_SINGLE │  58645 │              968227851 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ visitor     │ SHARED3    │  42819 │            -1374707706 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ visitor     │ TAXI       │   9132 │             1518431309 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ visitor     │ SHARED2    │   5661 │            -2025318090 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ vis